# 02 — Feature Engineering
Reads the raw `.parquet` files produced by `01_fetch_training_data.ipynb` and:
1. Computes all **11 normalised features** (matching `state_builder.py` exactly).
2. Builds **sliding window sequences** of length `LOOKBACK_BARS=60`.
3. Performs a **time-ordered 80/20 train/val split** (no shuffle — avoids lookahead bias).
4. Saves each ticker's arrays as `{TICKER}_train.npz` and `{TICKER}_val.npz`.

**Input:**  `/content/drive/MyDrive/algo_trader/data/raw/{TICKER}.parquet`  
**Output:** `/content/drive/MyDrive/algo_trader/data/features/{TICKER}_{split}.npz`

In [ ]:
!pip install -q pyarrow pandas numpy pytz tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
RAW_DIR  = '/content/drive/MyDrive/algo_trader/data/raw'
FEAT_DIR = '/content/drive/MyDrive/algo_trader/data/features'
os.makedirs(FEAT_DIR, exist_ok=True)
print('Directories ready ✓')

In [ ]:
# Clone the repo (or adjust to your Drive path if already cloned)
REPO_URL = 'https://github.com/rohanpatrick568/deepscalper_copilot.git'
REPO_DIR = '/content/deepscalper_copilot'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

# Pointing to the 'colab' directory allows us to import the 'deepscalper' package inside it
import sys
COLAB_DIR = os.path.join(REPO_DIR, 'algo_trader', 'colab')
if COLAB_DIR not in sys.path:
    sys.path.insert(0, COLAB_DIR)

print('Repo path updated for deepscalper package ✓')

In [ ]:
# Hyperparameters — must match config.py
LOOKBACK_BARS = 60
TRAIN_FRAC    = 0.80

SP100_TICKERS = [
    'AAPL','MSFT','AMZN','NVDA','GOOGL','GOOG','META','TSLA','BRK.B','UNH',
    'LLY','JPM','V','AVGO','XOM','MA','COST','PG','JNJ','HD',
    'ABBV','ORCL','BAC','WMT','NFLX','KO','CRM','CVX','MRK','AMD',
    'CSCO','PEP','ACN','LIN','TMO','MCD','ABT','IBM','GE','TXN',
    'PM','GS','ISRG','CAT','AXP','SPGI','AMGN','RTX','PFE','BKNG',
    'DHR','MS','INTU','BLK','T','VRTX','HON','NEE','UNP','SYK',
    'C','LOW','TJX','ADP','GILD','DE','PANW','BMY','AMAT','MDT',
    'PLD','SBUX','ADI','TMUS','ETN','SCHW','CB','MMC','BA','SO',
    'MO','WFC','UPS','CI','MDLZ','DUK','CL','INTC','REGN','PH',
    'EOG','SLB','ELV','APD','MCK','COF','ZTS','BSX','GEV','CME',
]
print(f'Processing {len(SP100_TICKERS)} tickers, LOOKBACK={LOOKBACK_BARS}')

In [ ]:
import numpy as np
import pandas as pd
import pytz
from tqdm.notebook import tqdm

# Import the shared feature pipeline
# Fixed the import path to match the directory structure added to sys.path
from deepscalper.utils import compute_features, compute_day_starts

def build_windows(
    features: np.ndarray,
    lookback: int,
) -> np.ndarray:
    """Slide a window of `lookback` bars over the feature matrix.

    Returns:
        Array of shape (n_windows, lookback, n_features) where
        window[i] = features[i : i + lookback].
    """
    n = len(features)
    if n < lookback:
        return np.empty((0, lookback, features.shape[1]), dtype=np.float32)
    indices = np.arange(lookback)[None, :] + np.arange(n - lookback + 1)[:, None]
    return features[indices]


summary = []
skipped = []

for ticker in tqdm(SP100_TICKERS, desc='Feature engineering'):
    raw_path   = f'{RAW_DIR}/{ticker}.parquet'
    train_path = f'{FEAT_DIR}/{ticker}_train.npz'
    val_path   = f'{FEAT_DIR}/{ticker}_val.npz'

    if os.path.exists(train_path) and os.path.exists(val_path):
        print(f'{ticker}: already processed — skipping.')
        skipped.append(ticker)
        continue

    if not os.path.exists(raw_path):
        print(f'WARNING: {raw_path} not found — skipping {ticker}.')
        skipped.append(ticker)
        continue

    # Load raw bars
    bars = pd.read_parquet(raw_path)
    bars.columns = [c.lower() for c in bars.columns]
    bars = bars[['open', 'high', 'low', 'close', 'volume']].astype(float)

    # Compute 11-feature matrix
    feat_matrix = compute_features(bars)   # (n_bars, 11)

    # Build sliding windows — shape (n_windows, 60, 11)
    windows = build_windows(feat_matrix, LOOKBACK_BARS)

    if len(windows) < 100:
        print(f'WARNING: {ticker} has only {len(windows)} windows — skipping.')
        skipped.append(ticker)
        continue

    # Time-ordered 80/20 split — NO shuffle to avoid lookahead bias
    split_idx = int(len(windows) * TRAIN_FRAC)
    X_train = windows[:split_idx]
    X_val   = windows[split_idx:]

    # Detect day boundaries for the environment's reset() method
    day_starts_all   = compute_day_starts(bars.index)
    # Clip to valid window indices
    train_day_starts = [d for d in day_starts_all if d < split_idx]
    val_day_starts   = [max(0, d - split_idx) for d in day_starts_all if d >= split_idx]

    np.savez_compressed(
        train_path,
        X=X_train.astype(np.float32),
        day_starts=np.array(train_day_starts, dtype=np.int32),
    )
    np.savez_compressed(
        val_path,
        X=X_val.astype(np.float32),
        day_starts=np.array(val_day_starts, dtype=np.int32),
    )

    summary.append({
        'ticker'       : ticker,
        'total_windows': len(windows),
        'train'        : len(X_train),
        'val'          : len(X_val),
        'train_days'   : len(train_day_starts),
        'val_days'     : len(val_day_starts),
    })

print('\n=== FEATURE ENGINEERING SUMMARY ===')
if summary:
    print(pd.DataFrame(summary).to_string(index=False))
if skipped:
    print(f'\nSkipped: {skipped}')
print('\nDone ✅')